In [0]:
# ============================================================
# SILVER LAYER FINAL VALIDATION
# ============================================================

silver_path = "abfss://silver@utilitiesmaintainence.dfs.core.windows.net/maintenance"

silver_df = spark.read.format("delta").load(silver_path)

print("========== SILVER TABLE VALIDATION ==========")

# 1. Record count
record_count = silver_df.count()
print(f"Silver record count: {record_count}")

# 2. Schema
print("\n========== SCHEMA ==========")
silver_df.printSchema()

# 3. Sample records
print("\n========== SAMPLE DATA ==========")
display(silver_df.limit(10))

# 4. Duplicate work_order_id check
duplicate_count = (
    silver_df
    .groupBy("work_order_id")
    .count()
    .filter("count > 1")
    .count()
)

print(f"\nDuplicate work_order_id values: {duplicate_count}")

# 5. Null work_order_id check
null_work_order_id = silver_df.filter(
    "work_order_id IS NULL"
).count()

print(f"Null work_order_id values: {null_work_order_id}")

# 6. Check important columns
required_columns = [
    "work_order_id",
    "asset_id",
    "technician_id",
    "location_id",
    "maintenance_type",
    "maintenance_category",
    "planned_flag",
    "failure_type",
    "failure_severity",
    "priority",
    "work_order_status",
    "resolution_code",
    "created_date",
    "scheduled_date",
    "completed_date",
    "downtime_hours",
    "labor_hours",
    "parts_used",
    "parts_cost",
    "labor_cost",
    "total_cost",
    "source_system",
    "updated_at"
]

missing_columns = [
    c for c in required_columns
    if c not in silver_df.columns
]

print(f"\nMissing required columns: {missing_columns}")

# 7. Final result
if (
    record_count > 0
    and duplicate_count == 0
    and null_work_order_id == 0
    and len(missing_columns) == 0
):
    print("\n✅ SILVER LAYER VALIDATION PASSED")
    print("✅ Silver layer is ready for Gold implementation.")
else:
    print("\n❌ SILVER LAYER VALIDATION FAILED")
    print("Fix the above issues before moving to Gold.")

========== SILVER TABLE VALIDATION ==========
Silver record count: 3600

========== SCHEMA ==========
root
 |-- work_order_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- technician_id: string (nullable = true)
 |-- location_id: string (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- maintenance_category: string (nullable = true)
 |-- planned_flag: string (nullable = true)
 |-- failure_type: string (nullable = true)
 |-- failure_severity: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- work_order_status: string (nullable = true)
 |-- resolution_code: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- scheduled_date: timestamp (nullable = true)
 |-- completed_date: timestamp (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- labor_hours: double (nullable = true)
 |-- parts_used: integer (nullable = true)
 |-- parts_cost: double (nullable = true)
 |-- labor_cost: double (null

work_order_id,asset_id,technician_id,location_id,maintenance_type,maintenance_category,planned_flag,failure_type,failure_severity,priority,work_order_status,resolution_code,created_date,scheduled_date,completed_date,downtime_hours,labor_hours,parts_used,parts_cost,labor_cost,total_cost,source_system,updated_at
WO-202604-01801,AST-00499,TECH-0029,LOC-0037,EMERGENCY,INSPECTION,N,CORROSION,HIGH,CRITICAL,COMPLETED,COMPONENT REPLACED,2026-04-27T07:54:19.000Z,2026-04-27T10:54:19.000Z,2026-04-27T17:57:23.000Z,14.68,29.83,5,9634.28,1402.05,11036.33,EAM,2026-04-27T17:57:23.000Z
WO-202604-01802,AST-00498,TECH-0054,LOC-0019,EMERGENCY,MECHANICAL,N,CONTROL SYSTEM FAULT,CRITICAL,CRITICAL,COMPLETED,ADJUSTED,2026-04-07T12:10:51.000Z,2026-04-07T14:10:51.000Z,2026-04-08T08:31:03.000Z,20.48,15.46,4,10268.66,955.18,11223.84,CMMS,2026-04-08T08:31:03.000Z
WO-202604-01803,AST-00479,TECH-0019,LOC-0021,PREVENTIVE,CONTROL SYSTEM,Y,NO FAILURE,NONE,MEDIUM,CLOSED,PREVENTIVE MAINTENANCE COMPLETED,2026-04-03T02:06:29.000Z,2026-04-05T02:06:29.000Z,2026-04-05T04:15:46.000Z,3.91,9.31,4,2975.72,730.77,3706.49,EAM,2026-04-05T04:15:46.000Z
WO-202604-01804,AST-00439,TECH-0081,LOC-0032,CORRECTIVE,CONTROL SYSTEM,N,INSULATION DAMAGE,MEDIUM,MEDIUM,COMPLETED,PREVENTIVE MAINTENANCE COMPLETED,2026-04-25T09:24:15.000Z,2026-04-28T09:24:15.000Z,2026-04-28T13:06:59.000Z,7.94,15.79,8,9143.58,1209.94,10353.52,CMMS,2026-04-28T13:06:59.000Z
WO-202604-01805,AST-00089,TECH-0033,LOC-0066,PREVENTIVE,ELECTRICAL,Y,NO FAILURE,NONE,HIGH,COMPLETED,REPAIRED,2026-04-15T23:13:20.000Z,2026-04-22T23:13:20.000Z,2026-04-23T02:36:48.000Z,7.8,10.24,7,2658.24,805.98,3464.22,CMMS,2026-04-23T02:36:48.000Z
WO-202604-01806,AST-00415,TECH-0016,LOC-0007,CORRECTIVE,MECHANICAL,N,VOLTAGE FLUCTUATION,MEDIUM,CRITICAL,CLOSED,ADJUSTED,2026-04-09T11:41:04.000Z,2026-04-16T11:41:04.000Z,2026-04-16T17:08:56.000Z,22.2,5.17,1,9215.31,503.93,9719.24,EAM,2026-04-16T17:08:56.000Z
WO-202604-01807,AST-00371,TECH-0065,LOC-0029,INSPECTION,ELECTRICAL,Y,NO FAILURE,NONE,MEDIUM,IN PROGRESS,null,2026-04-03T04:16:25.000Z,2026-04-07T04:16:25.000Z,null,1.34,16.08,1,239.2,1440.02,1679.22,CMMS,2026-04-07T05:51:12.000Z
WO-202604-01808,AST-00324,TECH-0054,LOC-0022,CORRECTIVE,CONTROL SYSTEM,N,CORROSION,LOW,LOW,COMPLETED,REPAIRED,2026-04-01T13:26:47.000Z,2026-04-06T13:26:47.000Z,2026-04-07T14:05:40.000Z,29.89,22.0,7,6151.86,2085.34,8237.2,FIELD_SERVICE,2026-04-07T14:05:40.000Z
WO-202604-01809,AST-00490,TECH-0090,LOC-0046,PREVENTIVE,MECHANICAL,Y,NO FAILURE,NONE,LOW,IN PROGRESS,null,2026-04-21T01:24:21.000Z,2026-04-22T01:24:21.000Z,null,9.5,9.7,3,1369.53,393.23,1762.76,EAM,2026-04-22T03:01:13.000Z
WO-202604-01810,AST-00213,TECH-0006,LOC-0061,PREVENTIVE,ELECTRICAL,Y,NO FAILURE,NONE,HIGH,COMPLETED,COMPONENT REPLACED,2026-04-01T08:48:03.000Z,2026-04-08T08:48:03.000Z,2026-04-08T10:59:12.000Z,2.82,15.66,2,2792.23,1016.92,3809.15,EAM,2026-04-08T10:59:12.000Z



Duplicate work_order_id values: 0
Null work_order_id values: 0

Missing required columns: []

✅ SILVER LAYER VALIDATION PASSED
✅ Silver layer is ready for Gold implementation.
